In [9]:
d_model = 64        # hidden size
history_len = 30
forecast_len = 7


GRN (Gated Residual Network)

Encounters: Inside the VSN.
What it does: Smart non-linear processing.

Action: It's a building block. It takes an input vector and:
Tries to process it with dense layers + ELU (non-linearity).
Uses a Gate (sigmoid) to decide: "Should I use this processed version, or just keep the original input?"

Why: It allows the network to learn complex patterns when needed, or just pass simple signals through untouched.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)

        self.gate = nn.Linear(output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)

        self.project_residual = (
            nn.Linear(input_dim, output_dim)
            if input_dim != output_dim else None
        )

    def forward(self, x):
        residual = x if self.project_residual is None else self.project_residual(x)

        x = F.elu(self.fc1(x))
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)

        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)


VariableSelectionNetwork (VSN)
Called Third: self.past_vsn(...) and self.future_vsn(...)

Role: The Smart Filter.

Action: It looks at the separate feature embeddings from the FeatureEncoder and asks: "Which of these features are actually useful right now?"
It uses a Weight Generator (a sub-network) to assign an importance score (e.g., Price: 0.8, Temp: 0.1, Noise: 0.1).
It creates a weighted combination of the features.

Result: A single, clean vector for each time step containing only the relevant information.
Sub-class used: GRN (Gated Residual Network)
Used by the VSN to process features and generate weights.
It allows the network to apply non-linear processing (using ELU activation) or skip processing entirely (using a Gate), giving it flexibility for both simple and complex patterns.

In [11]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()

        self.num_features = num_features

        # One GRN per feature
        self.feature_grns = nn.ModuleList([
            GRN(d_model, d_model) for _ in range(num_features)
        ])

        # Weight generator
        self.weight_grn = GRN(d_model * num_features, d_model)
        self.weight_fc = nn.Linear(d_model, num_features)

    def forward(self, x):
        """
        x: [B, T, num_features, d_model]
        """
        B, T, F, D = x.shape

        # Transform each feature
        transformed = []
        for i in range(F):
            transformed.append(self.feature_grns[i](x[:, :, i, :]))

        transformed = torch.stack(transformed, dim=2)  # [B, T, F, D]

        # Compute weights
        flat = transformed.reshape(B, T, F * D)
        weights = self.weight_grn(flat)
        weights = self.weight_fc(weights)
        weights = torch.softmax(weights, dim=-1)  # [B, T, F]

        # Weighted sum
        output = torch.sum(transformed * weights.unsqueeze(-1), dim=2)

        return output


StaticEncoder
Called First: self.static_encoder(static)

Role: Metadata Embedder.

Action: It takes your unchanging "static" data (like Store ID or Item ID) and transforms it into a feature vector (d_model).
Why: This gives the model "context." The model now knows what it is predicting for, before it even looks at the time-series history.

In [12]:
class StaticEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)

    def forward(self, x):
        return self.encoder(x)


FeatureEncoder 

What it does: Independent projecting.

Action: It takes your time-series data (e.g., 4 features like Price, Temp, Sales). It has 4 separate tiny distinct neural networks (Linear layers) inside it.
Feature 1 goes through Network 1.
Feature 2 goes through Network 2.
...

Output: [Batch, Time, 4, 64] — Each feature is now a rich vector of size 64, but they are still separate.

In [13]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [14]:
import torch.nn as nn


class TemporalAttention(nn.Module):
    """
    Multi-head self-attention across the time dimension.

    Receives the concatenated past + future VSN output of shape [B, T, D],
    lets every time step attend to every other time step, then applies a
    residual connection and LayerNorm for training stability.
    """

    def __init__(self, d_model: int, num_heads: int = 4):
        super().__init__()
        # batch_first=True so tensors are [B, T, D] throughout
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]  –  output from the VSN stage
        Returns:
            [B, T, D]     –  context-enriched representation
        """
        attn_out, _ = self.attn(x, x, x)       # self-attention (Q = K = V = x)
        return self.layer_norm(x + attn_out)    # residual + norm


MinimalTFT (The Conductor)

Role: This is the main class that orchestrates the entire process.

Action: It accepts the raw data (static, past, future) and passes it stage-by-stage through the other components. It manages the flow from encoding $\to$ variable selection $\to$ temporal attention $\to$ prediction.

In [15]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 static_dim,
                 past_dim,
                 future_dim,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        # Encoders
        self.static_encoder = StaticEncoder(static_dim, d_model)
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # Variable selection
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # Attention
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # Output head
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, static, past, future):
        """
        static: [B, S]
        past:   [B, T_hist, O]
        future: [B, T_hist + T_fut, K]
        """

        B, T_hist, O = past.shape
        T_total = future.shape[1]

        # Encode
        static_emb = self.static_encoder(static)

        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)


        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine timeline
        temporal_input = torch.cat([past_sel, future_sel[:, T_hist:]], dim=1)

        # Attention
        attn_out = self.temporal_attn(temporal_input)

        # Forecast
        forecast = self.output_layer(attn_out[:, -forecast_len:])
        return forecast.squeeze(-1)


In [16]:
B = 4
static = torch.randn(B, 5)
past = torch.randn(B, 30, 1)
future = torch.randn(B, 37, 4)

model = MinimalTFT(5, 1, 4)
out = model(static, past, future)

print(out.shape)  # [B, 7]


torch.Size([4, 7])


Raw Inputs
[B, T, F]
    ->
FeatureEncoder
[B, T, F, D]
    ->
VariableSelectionNetwork
[B, T, D]
    ->
TemporalAttention
[B, T, D]
    ->
Output Head
[B, T_fut]


In [17]:
import torch
import numpy as np

def generate_batch(batch_size=32, history=30, forecast=7):
    t_total = history + forecast
    
    base = torch.randn(batch_size, 1) * 5
    
    time = torch.arange(t_total).float()
    season = torch.sin(2 * np.pi * time / 7)

    sales = base + season + 0.1 * torch.randn(batch_size, t_total)
    
    past = sales[:, :history].unsqueeze(-1)
    future_target = sales[:, history:]
    
    # known future feature: day-of-week sine signal
    known = season.unsqueeze(0).repeat(batch_size, 1)
    known = known.unsqueeze(-1)
    
    static = torch.randn(batch_size, 5)
    
    return static, past, known, future_target

In [18]:
model = MinimalTFT(5, 1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch()

    optimizer.zero_grad()
    output = model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())

0 4.1877760887146
50 0.48675107955932617
100 0.2561398148536682
150 0.23938670754432678
200 0.34661880135536194
250 0.14848682284355164
300 0.10487527400255203
350 0.17743287980556488
400 0.17770935595035553
450 0.10030946880578995


In [19]:
class LSTMForecast(nn.Module):
    def __init__(self, input_dim=1, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 7)

    def forward(self, past):
        _, (h, _) = self.lstm(past)
        return self.fc(h.squeeze(0))